In [ ]:
import shutil
import tempfile

import ansys.fluent.core as pyfluent
from ansys.fluent.core import examples

wing_spaceclaim_file, wing_intermediary_file = [
    examples.download_file(CAD_file, "pyfluent/external_compressible")
    for CAD_file in ["wing.scdoc", "wing.pmdb"]
]


# Launch Fluent
# ~~~~~~~~~~~~~
# Launch Fluent as a service in meshing mode with double precision running on
# four processors and print Fluent version.

meshing_session = pyfluent.launch_fluent(
    precision="double",
    processor_count=4,
    mode="meshing",
)
print(meshing_session.get_fluent_version())

tmpdir = tempfile.mkdtemp()
meshing_session.preferences.MeshingWorkflow.TempFolder = tmpdir

# Initialize workflow
# ~~~~~~~~~~~~~~~~~~~
# Initialize the watertight geometry meshing workflow.

meshing_session.workflow.InitializeWorkflow(WorkflowType="Watertight Geometry")


#
# Import CAD and set length units
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import the CAD geometry and set the length units to inches.
geo_import = meshing_session.workflow.TaskObject["Import Geometry"]
geo_import.Arguments.set_state(
    {
        "FileName": wing_intermediary_file,
    }
)

geo_import.Execute()


# Add local sizing controls to the faceted geometry.
local_sizing = meshing_session.workflow.TaskObject["Add Local Sizing"]
local_sizing.Arguments.set_state(
    {
        "AddChild": "yes",
        "BOIControlName": "wing-facesize",
        "BOIFaceLabelList": ["wing_bottom", "wing_top"],
        "BOISize": 10,
    }
)

local_sizing.AddChildAndUpdate()

local_sizing.Arguments.set_state(
    {
        "AddChild": "yes",
        "BOIControlName": "wing-ege-facesize",
        "BOIFaceLabelList": ["wing_edge"],
        "BOISize": 2,
    }
)

local_sizing.AddChildAndUpdate()

local_sizing.Arguments.set_state(
    {
        "AddChild": "yes",
        "BOIControlName": "boi_1",
        "BOIExecution": "Body Of Influence",
        "BOIFaceLabelList": ["wing-boi"],
        "BOISize": 5,
    }
)

local_sizing.AddChildAndUpdate()


# Generate the surface mash.
surface_mesh_gen = meshing_session.workflow.TaskObject["Generate the Surface Mesh"]
surface_mesh_gen.Arguments.set_state(
    {"CFDSurfaceMeshControls": {"MaxSize": 1000, "MinSize": 2}}
)

surface_mesh_gen.Execute()


# Describe geometry
# ~~~~~~~~~~~~~~~~~
# Describe geometry and define the fluid region.
describe_geo = meshing_session.workflow.TaskObject["Describe Geometry"]
describe_geo.UpdateChildTasks(SetupTypeChanged=False)

describe_geo.Arguments.set_state(
    {"SetupType": "The geometry consists of only fluid regions with no voids"}
)

describe_geo.UpdateChildTasks(SetupTypeChanged=True)

describe_geo.Execute()


# Update boundaries
# ~~~~~~~~~~~~~~~~~
# Update the boundaries.

meshing_session.workflow.TaskObject["Update Boundaries"].Execute()


# Update regions
# ~~~~~~~~~~~~~~
# Update the regions.

meshing_session.workflow.TaskObject["Update Regions"].Execute()

# Add boundary layers
# ~~~~~~~~~~~~~~~~~~~
# Add boundary layers, which consist of setting properties for the
# boundary layer mesh.
add_boundary_layer = meshing_session.workflow.TaskObject["Add Boundary Layers"]
add_boundary_layer.Arguments.set_state({"NumberOfLayers": 12})

add_boundary_layer.AddChildAndUpdate()


# Generate volume mesh
# ~~~~~~~~~~~~~~~~~~~~
# Generate the volume mesh, which consists of setting properties for the
# volume mesh.
volume_mesh_gen = meshing_session.workflow.TaskObject["Generate the Volume Mesh"]
volume_mesh_gen.Arguments.set_state(
    {
        "VolumeFill": "poly-hexcore",
        "VolumeFillControls": {"HexMaxCellLength": 512},
        "VolumeMeshPreferences": {
            "CheckSelfProximity": "yes",
            "ShowVolumeMeshPreferences": True,
        },
    }
)

volume_mesh_gen.Execute()


# Check mesh in meshing mode
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# Check the mesh in meshing mode.

meshing_session.tui.mesh.check_mesh()


# Save mesh file
# ~~~~~~~~~~~~~~
# Save the mesh file (``wing.msh.h5``).

meshing_session.meshing.File.WriteMesh(FileName="wing.msh.h5")


# Solve and postprocess


solver_session = meshing_session.switch_to_solver()


# Check mesh in solver mode
# ~~~~~~~~~~~~~~~~~~~~~~~~~


solver_session.settings.mesh.check()


# Define model
# ~~~~~~~~~~~~
# Set the k-w sst turbulence model.

# model : k-omega
# k-omega model : sst

viscous = solver_session.settings.setup.models.viscous

viscous.model = "k-omega"
viscous.k_omega_model = "sst"


# Define materials


air = solver_session.settings.setup.materials.fluid["air"]

air.density.option = "ideal-gas"

air.viscosity.option = "sutherland"

air.viscosity.sutherland.option = "three-coefficient-method"

air.viscosity.sutherland.reference_viscosity = 1.716e-05

air.viscosity.sutherland.reference_temperature = 273.11

air.viscosity.sutherland.effective_temperature = 110.56


# Boundary Conditions
# ~~~~~~~~~~~~~~~~~~~
# Set the boundary conditions for ``pressure_farfield``.

# gauge pressure : 0 [Pa]
# mach number : 0.8395
# temperature : 255.56 [K]
# x-component of flow direction : 0.998574
# z-component of flow direction : 0.053382
# turbulent intensity : 5 [%]
# turbulent viscosity ratio : 10

pressure_farfield = (
    solver_session.settings.setup.boundary_conditions.pressure_far_field[
        "pressure_farfield"
    ]
)

pressure_farfield.momentum.gauge_pressure = 0

pressure_farfield.momentum.mach_number = 0.8395

pressure_farfield.thermal.temperature = 255.56

pressure_farfield.momentum.flow_direction[0] = 0.998574

pressure_farfield.momentum.flow_direction[2] = 0.053382

pressure_farfield.turbulence.turbulent_intensity = 0.05

pressure_farfield.turbulence.turbulent_viscosity_ratio = 10


# Operating Conditions
# ~~~~~~~~~~~~~~~~~~~~
# Set the operating conditions.

# operating pressure : 80600 [Pa]

solver_session.settings.setup.general.operating_conditions.operating_pressure = 80600


# Initialize flow field
# ~~~~~~~~~~~~~~~~~~~~~
# Initialize the flow field using hybrid initialization.

solver_session.settings.solution.initialization.hybrid_initialize()

# Save case file

solver_session.settings.file.write(
    file_name="external_compressible.cas.h5", file_type="case"
)

# Solve for 25 iterations
# ~~~~~~~~~~~~~~~~~~~~~~~~
# Solve for 25 iterations (100 iterations is recommended, however for this example 25 is sufficient).

solver_session.settings.solution.run_calculation.iterate(iter_count=25)

# Write final case file and data
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Write the final case file and the data.

solver_session.settings.file.write(
    file_name="external_compressible1.cas.h5", file_type="case"
)


# Close Fluent
# ~~~~~~~~~~~~
# Close Fluent.

solver_session.exit()

shutil.rmtree(tmpdir, ignore_errors=True)
shutil.rmtree("wing_workflow_files", ignore_errors=True)
